<a href="https://colab.research.google.com/github/haoyuz1/Australian-Shopping-Centres/blob/main/Automating_the_VIX_ETNs_Volatility_Strategy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Automating the VIX ETNs Volatility Strategy**

---

**Authors:**  
[**Carlo Zarattini**](mailto:Carlo@concretumgroup.com) – *First author of the research paper*  
[**Mohamed Gabriel**](mailto:Mohamed@concretumgroup.com) – *Implementation and automation under Carlo's supervision*

---

This notebook implements the volatility strategy from the research paper:  
**“The Volatility Edge: A Practical Guide For VIX ETNs Trading”**  
developed and published by **Concretum Group**.

The code connects to Interactive Brokers (TWS) and automates daily signal generation and trade execution based on SPY returns, VIX term structure, and realized volatility.

<div style="background-color: #fff3cd; border: 1px solid #ffeeba; border-radius: 4px; padding: 12px; margin: 20px 0;">
  <p style="margin: 0; color: #856404;"><strong>⚠️ Local Execution Only:</strong> This strategy <em>does not work</em> on Google Colab. It requires a local Python environment with <strong>IBKR's Trader Workstation (TWS)</strong> running and socket client access enabled.</p>
</div>

---

📧 **Contact:**  
[info@concretumgroup.com](mailto:info@concretumgroup.com)  
🌐 **Website:** [www.concretumgroup.com](https://www.concretumgroup.com)

---
## Disclaimer

This notebook is provided as a **template implementation only**, for **educational and research purposes**.

You are fully responsible for reviewing, modifying, and validating all aspects of this code before using it in any live trading environment. **We do not guarantee performance or accuracy**, and **we are not liable for any trading losses or damages** resulting from its use.

Please consult a qualified financial advisor and ensure full understanding of the risks involved before applying this strategy in practice.

---
## Getting Started

Before running the strategy:
- Install required packages (see Cell 0)
- Launch IBKR TWS (paper account)
- Ensure socket client settings are enabled
- Adjust account and execution parameters in the config cells (Cell 5)


## Cell 0: Install Required Packages


In [1]:
!pip install ib_async
!pip install ipywidgets
!pip install pandas
!pip install pytz
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.4 MB/s eta 0:00:00


## Cell 1: Setup and Configuration


In [2]:
from ib_async import util, IB, Stock, Index

def setup_strategy(config):
    """Setup IB connection and define contracts"""
    # Setup
    util.startLoop()
    ib = IB()
    ib.connect('127.0.0.1', 7497, clientId=1)

    # Define instruments
    spy = Stock('SPY', 'SMART', 'USD')
    vix = Index('VIX', 'CBOE', 'USD')
    vix3m = Index('VIX3M', 'CBOE', 'USD')
    vxx = Stock('VXX', 'SMART', 'USD')
    svxy = Stock('SVXY', 'SMART', 'USD')
    ib.qualifyContracts(spy, vix, vix3m, vxx, svxy)

    contracts = {
        'spy': spy,
        'vix': vix,
        'vix3m': vix3m,
        'vxx': vxx,
        'svxy': svxy
    }

    print("Connected and contracts qualified")
    return ib, contracts

print("Setup function defined")

Setup function defined


## Cell 2: Timing and Data Collection


In [3]:
import pytz
import time as time_module
from datetime import datetime
import pandas as pd
from ib_async import util

def collect_market_data(ib, config, contracts):
    """Collect market data with timing logic"""

    ny_tz = pytz.timezone('America/New_York')
    current_ny_time = datetime.now(ny_tz)
    current_time = current_ny_time.time()

    print(f"Current time: {current_ny_time.strftime('%H:%M:%S')} NY")

    if current_time < config['WAIT_UNTIL_TIME']:
        wait_until = datetime.combine(current_ny_time.date(), config['WAIT_UNTIL_TIME'])
        wait_until = ny_tz.localize(wait_until)
        wait_seconds = (wait_until - current_ny_time).total_seconds()

        print(f"Waiting until {config['WAIT_UNTIL_TIME']} NY ({wait_seconds:.0f} seconds)...")
        time_module.sleep(wait_seconds)
        print(f"Now {config['WAIT_UNTIL_TIME']} NY - proceeding")

    execution_time = datetime.now(ny_tz)

    # Get SPY daily data
    spy_daily = ib.reqHistoricalData(contracts['spy'], '', '30 D', '1 day', 'ADJUSTED_LAST', True)
    spy_daily_df = util.df(spy_daily)

    # Convert timestamps
    spy_daily_dt = pd.to_datetime(spy_daily_df['date'])
    if spy_daily_dt.dt.tz is None:
        spy_daily_df['date_ny'] = spy_daily_dt.dt.tz_localize(ny_tz)
    else:
        spy_daily_df['date_ny'] = spy_daily_dt.dt.tz_convert(ny_tz)

    # Get today's minute data
    spy_today = ib.reqHistoricalData(contracts['spy'], '', '1 D', '1 min', 'ADJUSTED_LAST', True)
    vix_today = ib.reqHistoricalData(contracts['vix'], '', '1 D', '1 min', 'ADJUSTED_LAST', True)
    vix3m_today = ib.reqHistoricalData(contracts['vix3m'], '', '1 D', '1 min', 'ADJUSTED_LAST', True)

    def convert_to_ny_time(df):
        dt = pd.to_datetime(df['date'])
        if dt.dt.tz is None:
            return dt.dt.tz_localize(ny_tz)
        else:
            return dt.dt.tz_convert(ny_tz)

    spy_today_df = util.df(spy_today)
    vix_df = util.df(vix_today)
    vix3m_df = util.df(vix3m_today)

    spy_today_df['time_ny'] = convert_to_ny_time(spy_today_df).dt.time
    vix_df['time_ny'] = convert_to_ny_time(vix_df).dt.time
    vix3m_df['time_ny'] = convert_to_ny_time(vix3m_df).dt.time

    def get_close_at_time(df, target_time):
        exact_match = df[df['time_ny'] == target_time]
        if len(exact_match) > 0:
            return exact_match['close'].iloc[0]

        before_target = df[df['time_ny'] <= target_time]
        if len(before_target) > 0:
            return before_target['close'].iloc[-1]

        return df['close'].iloc[0]

    spy_close_today = get_close_at_time(spy_today_df, config['TARGET_CLOSE_TIME'])
    vix_close = get_close_at_time(vix_df, config['TARGET_CLOSE_TIME'])
    vix3m_close = get_close_at_time(vix3m_df, config['TARGET_CLOSE_TIME'])

    print(f"SPY: ${spy_close_today:.2f}, VIX: {vix_close:.2f}, VIX3M: {vix3m_close:.2f}")

    return {
        'execution_time': execution_time,
        'spy_daily_df': spy_daily_df,
        'spy_close_today': spy_close_today,
        'vix_close': vix_close,
        'vix3m_close': vix3m_close
    }

print("Data collection function defined")

Data collection function defined


## Cell 3: Calculate Signals and Manage Positions


In [4]:
from ib_async import Order
import numpy as np

def calculate_signals_and_manage_positions(ib, config, contracts, market_data, current_positions):
    """Calculate signals using Strategy 4: eVRP + BoC + Sizing logic and manage VXX/SVXY positions"""

    spy_daily_df_modified = market_data['spy_daily_df'].copy()
    spy_daily_df_modified.iloc[-1, spy_daily_df_modified.columns.get_loc('close')] = market_data['spy_close_today']
    spy_daily_df_modified['spy_ret'] = spy_daily_df_modified['close'].pct_change()

    all_returns = spy_daily_df_modified['spy_ret'].dropna().tolist()
    last_10_returns = all_returns[-10:]

    realized_vol = np.std(last_10_returns, ddof=1) * np.sqrt(252) * 100

    print(f"10-day Realized Volatility: {realized_vol:.2f}%")

    eVRP = market_data['vix_close'] - realized_vol

    vix_below_vix3m = market_data['vix_close'] < market_data['vix3m_close']  # Contango
    vix_above_vix3m = market_data['vix_close'] > market_data['vix3m_close']  # Backwardation

    print(f"VIX: {market_data['vix_close']:.2f}")
    print(f"VIX3M: {market_data['vix3m_close']:.2f}")
    print(f"VIX < VIX3M (Contango): {vix_below_vix3m}")
    print(f"VIX > VIX3M (Backwardation): {vix_above_vix3m}")
    print(f"eVRP (expected Volatility Risk Premium): {eVRP:.2f}")

    # Strategy 4: eVRP + BoC + Sizing Logic
    if eVRP > 0 and vix_below_vix3m:
        # Condition 1: eVRP > 0 and VIX < VIX3M -> Long VIXSHORT with VIX%
        signal = -1
        vxx_weight = 0
        svxy_weight = (market_data['vix_close'] / 100) * 2  # VIX% × 2 (SVXY has 0.5x leverage)
        action = "LONG VIXSHORT (eVRP > 0, VIX < VIX3M)"
        primary_instrument = "SVXY"
        sizing_multiplier = 1.0

    elif eVRP < 0 and vix_below_vix3m:
        # Condition 2: eVRP < 0 and VIX < VIX3M -> Long VIXSHORT with 0.5 × VIX%
        signal = -1
        vxx_weight = 0
        svxy_weight = (market_data['vix_close'] / 100) * 0.5 * 2  # 0.5 × VIX% × 2 (SVXY has 0.5x leverage)
        action = "LONG VIXSHORT 0.5x (eVRP < 0, VIX < VIX3M)"
        primary_instrument = "SVXY"
        sizing_multiplier = 0.5

    elif eVRP < 0 and vix_above_vix3m:
        # Condition 3: eVRP < 0 and VIX > VIX3M -> Long VIXLONG with VIX%
        signal = 1
        vxx_weight = market_data['vix_close'] / 100  # VIX%
        svxy_weight = 0
        action = "LONG VIXLONG (eVRP < 0, VIX > VIX3M)"
        primary_instrument = "VXX"
        sizing_multiplier = 1.0

    else:
        # Condition 4: eVRP > 0 and VIX > VIX3M -> Stay in cash
        signal = 0
        vxx_weight = 0
        svxy_weight = 0
        action = "STAY IN CASH (eVRP > 0, VIX > VIX3M)"
        primary_instrument = "NONE"
        sizing_multiplier = 0.0

    print(f"\nSTRATEGY 4 SIGNAL:")
    print(f"Condition: {action}")
    print(f"Signal: {signal}")
    print(f"Primary Instrument: {primary_instrument}")
    print(f"Sizing Multiplier: {sizing_multiplier}")
    print(f"VXX Weight: {vxx_weight:.3f} ({vxx_weight*100:.1f}%)")
    print(f"SVXY Weight: {svxy_weight:.3f} ({svxy_weight*100:.1f}%)")

    # Get current prices
    vxx_ticker = ib.reqMktData(contracts['vxx'], '', False, False)
    svxy_ticker = ib.reqMktData(contracts['svxy'], '', False, False)
    ib.sleep(2)

    vxx_price = vxx_ticker.last if vxx_ticker.last and vxx_ticker.last > 0 else vxx_ticker.close
    svxy_price = svxy_ticker.last if svxy_ticker.last and svxy_ticker.last > 0 else svxy_ticker.close

    if not vxx_price or vxx_price <= 0:
        print("Could not get VXX price")
        vxx_price = None
    if not svxy_price or svxy_price <= 0:
        print("Could not get SVXY price")
        svxy_price = None

    print(f"\nCurrent Prices:")
    print(f"  VXX: ${vxx_price:.2f}" if vxx_price else "  VXX: Price unavailable")
    print(f"  SVXY: ${svxy_price:.2f}" if svxy_price else "  SVXY: Price unavailable")

    # Calculate target positions
    vxx_target_notional = vxx_weight * config['ACCOUNT_VALUE']
    svxy_target_notional = svxy_weight * config['ACCOUNT_VALUE']

    vxx_target_shares = int(vxx_target_notional / vxx_price) if vxx_price else 0
    svxy_target_shares = int(svxy_target_notional / svxy_price) if svxy_price else 0

    print(f"\nTarget Positions:")
    print(f"  VXX: {vxx_target_shares:,} shares (${vxx_target_notional:,.2f})")
    print(f"  SVXY: {svxy_target_shares:,} shares (${svxy_target_notional:,.2f})")

    # Calculate current weights
    current_vxx_value = current_positions['vxx_shares'] * vxx_price if vxx_price else 0
    current_svxy_value = current_positions['svxy_shares'] * svxy_price if svxy_price else 0

    current_vxx_weight = current_vxx_value / config['ACCOUNT_VALUE']
    current_svxy_weight = current_svxy_value / config['ACCOUNT_VALUE']

    print(f"\nCurrent Weights:")
    print(f"  VXX: {current_vxx_weight:.3f} ({current_vxx_weight*100:.1f}%)")
    print(f"  SVXY: {current_svxy_weight:.3f} ({current_svxy_weight*100:.1f}%)")

    # Calculate weight differences
    vxx_weight_diff = abs(vxx_weight - current_vxx_weight)
    svxy_weight_diff = abs(svxy_weight - current_svxy_weight)

    print(f"\nWeight Differences:")
    print(f"  VXX: {vxx_weight_diff:.3f} ({vxx_weight_diff*100:.1f}% diff)")
    print(f"  SVXY: {svxy_weight_diff:.3f} ({svxy_weight_diff*100:.1f}% diff)")

    # Use 2% tolerance as specified in Strategy 4 (±2% rebalance threshold)
    weight_tolerance = config.get('WEIGHT_TOLERANCE', 0.02)  # Default 2% for Strategy 4
    print(f"\nWeight Tolerance: {weight_tolerance:.3f} ({weight_tolerance*100:.1f}%)")

    # Calculate position changes needed
    vxx_change = vxx_target_shares - current_positions['vxx_shares']
    svxy_change = svxy_target_shares - current_positions['svxy_shares']

    print(f"\nCurrent Positions:")
    print(f"  VXX: {current_positions['vxx_shares']:,} shares (${current_vxx_value:,.2f})")
    print(f"  SVXY: {current_positions['svxy_shares']:,} shares (${current_svxy_value:,.2f})")

    print(f"\nPosition Changes Needed:")
    print(f"  VXX: {vxx_change:+,} shares")
    print(f"  SVXY: {svxy_change:+,} shares")

    # Apply tolerance filter and place orders
    orders_placed = []

    # VXX Order Logic
    if vxx_change != 0 and vxx_price:
        if vxx_weight_diff > weight_tolerance:
            vxx_action = "BUY" if vxx_change > 0 else "SELL"
            vxx_quantity = abs(vxx_change)
            print(f"\nPLACING VXX MOC ORDER (diff {vxx_weight_diff*100:.1f}% > {weight_tolerance*100:.1f}%):")
            print(f"Action: {vxx_action} {vxx_quantity:,} shares of VXX")

            vxx_order = Order(orderType="MOC", action=vxx_action, totalQuantity=vxx_quantity)
            vxx_trade = ib.placeOrder(contracts['vxx'], vxx_order)
            orders_placed.append(('VXX', vxx_action, vxx_quantity, vxx_trade))
            print(f"VXX Order placed: {vxx_trade}")
        else:
            print(f"\nSKIPPING VXX ORDER - Weight difference {vxx_weight_diff*100:.1f}% is within tolerance {weight_tolerance*100:.1f}%")

    # SVXY Order Logic
    if svxy_change != 0 and svxy_price:
        if svxy_weight_diff > weight_tolerance:
            svxy_action = "BUY" if svxy_change > 0 else "SELL"
            svxy_quantity = abs(svxy_change)
            print(f"\nPLACING SVXY MOC ORDER (diff {svxy_weight_diff*100:.1f}% > {weight_tolerance*100:.1f}%):")
            print(f"Action: {svxy_action} {svxy_quantity:,} shares of SVXY")

            svxy_order = Order(orderType="MOC", action=svxy_action, totalQuantity=svxy_quantity)
            svxy_trade = ib.placeOrder(contracts['svxy'], svxy_order)
            orders_placed.append(('SVXY', svxy_action, svxy_quantity, svxy_trade))
            print(f"SVXY Order placed: {svxy_trade}")
        else:
            print(f"\nSKIPPING SVXY ORDER - Weight difference {svxy_weight_diff*100:.1f}% is within tolerance {weight_tolerance*100:.1f}%")

    if not orders_placed:
        print("\nNo orders placed - all positions within tolerance or no changes needed")

    return {
        'realized_vol': realized_vol,
        'eVRP': eVRP,
        'vix_below_vix3m': vix_below_vix3m,
        'vix_above_vix3m': vix_above_vix3m,
        'signal': signal,
        'action': action,
        'primary_instrument': primary_instrument,
        'sizing_multiplier': sizing_multiplier,
        'vxx_weight': vxx_weight,
        'svxy_weight': svxy_weight,
        'vxx_price': vxx_price,
        'svxy_price': svxy_price,
        'vxx_target_shares': vxx_target_shares,
        'svxy_target_shares': svxy_target_shares,
        'vxx_target_notional': vxx_target_notional,
        'svxy_target_notional': svxy_target_notional,
        'vxx_change': vxx_change,
        'svxy_change': svxy_change,
        'orders_placed': orders_placed,
        'current_vxx_weight': current_vxx_weight,
        'current_svxy_weight': current_svxy_weight,
        'vxx_weight_diff': vxx_weight_diff,
        'svxy_weight_diff': svxy_weight_diff,
        'weight_tolerance': weight_tolerance,
    }

print("eVRP + BoC + Sizing signal calculation and position management function defined")

eVRP + BoC + Sizing signal calculation and position management function defined


## Cell 4: Log Results and Disconnect


In [5]:
import pandas as pd
import os

def log_results_and_cleanup(ib, config, market_data, trading_results, current_positions):
    """Log all data to CSV and disconnect"""

    log_data = {
        'timestamp': market_data['execution_time'].strftime('%Y-%m-%d %H:%M:%S'),
        'date': market_data['execution_time'].strftime('%Y-%m-%d'),
        'time_ny': market_data['execution_time'].strftime('%H:%M:%S'),
        'target_time': config['TARGET_CLOSE_TIME'].strftime('%H:%M:%S'),
        'spy_close': market_data['spy_close_today'],
        'vix_close': market_data['vix_close'],
        'vix3m_close': market_data['vix3m_close'],
        'realized_vol': trading_results['realized_vol'],
        'boc': trading_results['boc'],
        'vrp': trading_results['vrp'],
        'signal': trading_results['signal'],
        'action': trading_results['action'],
        'primary_instrument': trading_results['primary_instrument'],
        'current_vxx_weight': trading_results['current_vxx_weight'],
        'current_svxy_weight': trading_results['current_svxy_weight'],
        'current_vxx_weight_pct': trading_results['current_vxx_weight'] * 100,
        'current_svxy_weight_pct': trading_results['current_svxy_weight'] * 100,
        'vxx_weight_diff': trading_results['vxx_weight_diff'],
        'svxy_weight_diff': trading_results['svxy_weight_diff'],
        'vxx_weight_diff_pct': trading_results['vxx_weight_diff'] * 100,
        'svxy_weight_diff_pct': trading_results['svxy_weight_diff'] * 100,
        'weight_tolerance': trading_results['weight_tolerance'],
        'weight_tolerance_pct': trading_results['weight_tolerance'] * 100,
        'vxx_weight': trading_results['vxx_weight'],
        'svxy_weight': trading_results['svxy_weight'],
        'vxx_weight_pct': trading_results['vxx_weight'] * 100,
        'svxy_weight_pct': trading_results['svxy_weight'] * 100,
        'current_vxx_shares': current_positions['vxx_shares'],
        'current_svxy_shares': current_positions['svxy_shares'],
        'vxx_price': trading_results['vxx_price'],
        'svxy_price': trading_results['svxy_price'],
        'vxx_target_shares': trading_results['vxx_target_shares'],
        'svxy_target_shares': trading_results['svxy_target_shares'],
        'vxx_target_notional': trading_results['vxx_target_notional'],
        'svxy_target_notional': trading_results['svxy_target_notional'],
        'vxx_change': trading_results['vxx_change'],
        'svxy_change': trading_results['svxy_change'],
        'orders_placed_count': len(trading_results['orders_placed']),
        'orders_summary': str(trading_results['orders_placed']) if trading_results['orders_placed'] else 'None'
    }

    log_df = pd.DataFrame([log_data])

    if os.path.exists(config['LOG_FILE']):
        log_df.to_csv(config['LOG_FILE'], mode='a', header=False, index=False)
        print(f"\nData logged to {config['LOG_FILE']} (appended)")
    else:
        log_df.to_csv(config['LOG_FILE'], mode='w', header=True, index=False)
        print(f"\nData logged to {config['LOG_FILE']} (new file created)")

    print("\n=== LOGGED DATA ===")
    for key, value in log_data.items():
        if value is not None:
            if isinstance(value, float):
                print(f"{key}: {value:.4f}")
            else:
                print(f"{key}: {value}")

    ib.disconnect()
    print("Disconnected from IBKR")

    return log_data

print("Logging and cleanup function defined")

Logging and cleanup function defined


## Cell 5: Execute Complete Strategy
Run the entire VIX strategy pipeline by calling all functions in sequence.


In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import time

def create_strategy_input_interface():
    """Create a user-friendly input interface using ipywidgets"""
    print("=" * 60)
    print("VIX VOLATILITY STRATEGY - USER INPUT INTERFACE")
    print("=" * 60)

    closure_time_widget = widgets.Dropdown(
        options=[
            ('16:00 (4:00 PM)', 16),
            ('15:30 (3:30 PM)', 15.5),
            ('15:00 (3:00 PM)', 15),
            ('14:30 (2:30 PM)', 14.5),
            ('14:00 (2:00 PM)', 14),
            ('13:30 (1:30 PM)', 13.5),
            ('13:00 (1:00 PM)', 13),
            ('Custom', 'custom')
        ],
        value=16,
        description='Market Closure:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )

    custom_hour_widget = widgets.IntSlider(
        value=16,
        min=9,
        max=22,
        description='Hour:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px', display='none')
    )

    custom_minute_widget = widgets.IntSlider(
        value=0,
        min=0,
        max=59,
        step=1,
        description='Minute:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px', display='none')
    )

    account_value_widget = widgets.IntText(
        value=100000,
        description='Account Value ($):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )

    current_vxx_widget = widgets.IntText(
        value=0,
        description='Current VXX Shares:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )

    current_svxy_widget = widgets.IntText(
        value=0,
        description='Current SVXY Shares:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )

    weight_tolerance_widget = widgets.FloatSlider(
        value=1.0,
        min=0.1,
        max=5.0,
        step=0.1,
        description='Weight Tolerance (%):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )

    log_file_widget = widgets.Text(
        value='vix_strategy_results.csv',
        description='Log File Name:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )

    submit_button = widgets.Button(
        description='Execute Strategy',
        button_style='success',
        layout=widgets.Layout(width='200px', height='40px')
    )

    output_widget = widgets.Output()

    def on_closure_time_change(change):
        if change['new'] == 'custom':
            custom_hour_widget.layout.display = 'block'
            custom_minute_widget.layout.display = 'block'
        else:
            custom_hour_widget.layout.display = 'none'
            custom_minute_widget.layout.display = 'none'

    closure_time_widget.observe(on_closure_time_change, names='value')

    def calculate_times(market_closure_hour):
        """Calculate TARGET_CLOSE_TIME and WAIT_UNTIL_TIME based on market closure"""
        if isinstance(market_closure_hour, str):  # Custom option
            closure_hour = custom_hour_widget.value
            closure_minute = custom_minute_widget.value
        else:
            closure_hour = int(market_closure_hour)
            closure_minute = int((market_closure_hour % 1) * 60)

        closure_total_minutes = closure_hour * 60 + closure_minute

        target_total_minutes = closure_total_minutes - 16
        target_hour = target_total_minutes // 60
        target_minute = target_total_minutes % 60

        # WAIT_UNTIL_TIME: 15 minutes before closure + 5 seconds
        wait_total_minutes = closure_total_minutes - 15
        wait_hour = wait_total_minutes // 60
        wait_minute = wait_total_minutes % 60

        target_close_time = time(target_hour, target_minute)
        wait_until_time = time(wait_hour, wait_minute, 5)

        return target_close_time, wait_until_time

    def on_submit_click(button):
        with output_widget:
            clear_output()

            try:
                market_closure = closure_time_widget.value
                account_value = account_value_widget.value
                current_vxx = current_vxx_widget.value
                current_svxy = current_svxy_widget.value
                weight_tolerance = weight_tolerance_widget.value / 100  # Convert to decimal
                log_file = log_file_widget.value

                target_close_time, wait_until_time = calculate_times(market_closure)

                if market_closure == 'custom':
                    closure_display = f"{custom_hour_widget.value:02d}:{custom_minute_widget.value:02d}"
                else:
                    closure_hour = int(market_closure)
                    closure_minute = int((market_closure % 1) * 60)
                    closure_display = f"{closure_hour:02d}:{closure_minute:02d}"

                print("=" * 50)
                print("STRATEGY CONFIGURATION CONFIRMED")
                print("=" * 50)
                print(f"Market Closure Time: {closure_display}")
                print(f"Target Close Time: {target_close_time}")
                print(f"Wait Until Time: {wait_until_time}")
                print(f"Account Value: ${account_value:,}")
                print(f"Current VXX Shares: {current_vxx:,}")
                print(f"Current SVXY Shares: {current_svxy:,}")
                print(f"Weight Tolerance: {weight_tolerance:.1%}")
                print(f"Log File: {log_file}")
                print("=" * 50)

                config = {
                    'TARGET_CLOSE_TIME': target_close_time,
                    'WAIT_UNTIL_TIME': wait_until_time,
                    'ACCOUNT_VALUE': account_value,
                    'WEIGHT_TOLERANCE': weight_tolerance,
                    'LOG_FILE': log_file
                }

                current_positions = {
                    'vxx_shares': current_vxx,
                    'svxy_shares': current_svxy
                }

                print("\nExecuting strategy...")
                result = execute_strategy_with_config(config, current_positions)

                if result:
                    print("\nStrategy executed successfully!")
                else:
                    print("\nStrategy execution failed!")

            except Exception as e:
                print(f"\nError: {str(e)}")

    submit_button.on_click(on_submit_click)

    form_box = widgets.VBox([
        widgets.HTML("<h3>Market Configuration</h3>"),
        closure_time_widget,
        custom_hour_widget,
        custom_minute_widget,
        widgets.HTML("<br><h3>Account Settings</h3>"),
        account_value_widget,
        weight_tolerance_widget,
        widgets.HTML("<br><h3>Current Positions</h3>"),
        current_vxx_widget,
        current_svxy_widget,
        widgets.HTML("<br><h3>Logging</h3>"),
        log_file_widget,
        widgets.HTML("<br>"),
        submit_button,
        widgets.HTML("<br>"),
        output_widget
    ])

    display(form_box)

def execute_strategy_with_config(config, current_positions):
    """Execute the complete VIX strategy with provided configuration"""
    try:
        # Step 1: Setup
        print("\n1. SETUP AND CONFIGURATION")
        print("-" * 30)
        ib, contracts = setup_strategy(config)

        # Step 2: Data Collection
        print("\n2. DATA COLLECTION")
        print("-" * 30)
        market_data = collect_market_data(ib, config, contracts)

        # Step 3: Signal Calculation and Position Management
        print("\n3. SIGNAL CALCULATION AND POSITION MANAGEMENT")
        print("-" * 30)
        trading_results = calculate_signals_and_manage_positions(ib, config, contracts, market_data, current_positions)

        # Step 4: Logging and Cleanup
        print("\n4. LOGGING AND CLEANUP")
        print("-" * 30)
        log_data = log_results_and_cleanup(ib, config, market_data, trading_results, current_positions)

        print("\n" + "="*60)
        print("STRATEGY EXECUTION COMPLETED SUCCESSFULLY")
        print("="*60)

        return log_data

    except Exception as e:
        print(f"\nERROR: {e}")
        try:
            ib.disconnect()
            print("Disconnected from IBKR due to error")
        except:
            pass
        return None

print("Enhanced user interface functions defined")

create_strategy_input_interface()

Enhanced user interface functions defined
VIX VOLATILITY STRATEGY - USER INPUT INTERFACE
